# Machine Learning Final Project: 
## Attention-based Neurosymbolic Differentiable Rule Extractor (ANDRE)

**Creator**: Iman Sharifi, G46454668

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

def interpret_neuron_weights_entropy_(body_preds, pred_values, Weights, entropy_threshold=0.8):
    """
    Interpret the weights of neurons and return a list of 'b_{i}', 'not(b_{i})', or '1'.

    Args:
        body_preds (list): A list of predicate strings for each neuron, like ['b1', 'b2', ...] or ['b1(X1, X2)', 'b2(X3)', ...]
        pred_value: The predicted value to be used in interpretation. it is a scalar value between 0 and 1.
        weights (numpy.ndarray): A 2D numpy array where each row corresponds to the weights of a neuron.

    Returns:
        list: A list of strings ('b_{i}', 'not(b_{i})', or '1') for each neuron.
    """
    
    interpretations = []
    for i, weights in enumerate(Weights):
        weights_tensor = torch.tensor(weights, dtype=torch.float32)
        entropy = -torch.sum(weights_tensor * torch.log(weights_tensor + 1e-10)).item()
        # print(f"Neuron {i+1} weights: {weights}, Entropy: {entropy}")

        if entropy > entropy_threshold:
            interpretations.append('?')
            continue

        # Find predicate value of a given neuron
        pred_value = pred_values[i]
        body_pred = body_preds[i]

        # compute {pred_value*weights[0], (1-pred_value)*weights[1], weights[2]}
        new_weights = [pred_value * weights[0], (1 - pred_value) * weights[1], weights[2]]

        max_index = np.argmax(new_weights)
        if max_index == 0:
            interpretations.append(f'{body_pred}')
            continue
        elif max_index == 1:
            interpretations.append(f'not({body_pred})')
            continue
        else:
            interpretations.append('1')

    return interpretations

def interpret_neuron_weights_entropy(pred_values, Weights, entropy_threshold=0.8):
    """
    Interpret the weights of neurons and return a list of 'b_{i}', 'not(b_{i})', or '1',
    only if entropy is below a threshold to ensure confidence in prediction.

    Args:
        pred_values (numpy.ndarray): A 1D array with predicate values (between 0 and 1).
        Weights (list of list): A 2D list where each row is the softmaxed weights of a neuron.
        entropy_threshold (float): The threshold of entropy above which output is considered uncertain.

    Returns:
        list: A list of strings ('b_{i}', 'not(b_{i})', or '1') for each neuron, or '?' for uncertain.
    """
    interpretations = []
    for i, weights in enumerate(Weights):
        weights_tensor = torch.tensor(weights, dtype=torch.float32)
        entropy = -torch.sum(weights_tensor * torch.log(weights_tensor + 1e-10)).item()
        # print(f"Neuron {i+1} weights: {weights}, Entropy: {entropy}")

        if entropy > entropy_threshold:
            interpretations.append('?')
            continue

        pred_value = pred_values[i]
        new_weights = [pred_value * weights[0], (1 - pred_value) * weights[1], weights[2]]
        max_index = np.argmax(new_weights)

        if max_index == 0:
            interpretations.append(f'b{i+1}')
        elif max_index == 1:
            interpretations.append(f'not(b{i+1})')
        else:
            interpretations.append('1')

    return interpretations

def interpret_neuron_weights_(pred_values, Weights):
    """
    Interpret the weights of neurons and return a list of 'b_{i}', 'not(b_{i})', or '1'.

    Args:
        pred_value: The predicted value to be used in interpretation. it is a scalar value between 0 and 1.
        weights (numpy.ndarray): A 2D numpy array where each row corresponds to the weights of a neuron.

    Returns:
        list: A list of strings ('b_{i}', 'not(b_{i})', or '1') for each neuron.
    """
    

    interpretations = []
    for i, weights in enumerate(Weights):

        # Find predicate value of a given neuron
        pred_value = pred_values[i]

        # compute {pred_value*weights[0], (1-pred_value)*weights[1], weights[2]}
        new_weights = [pred_value * weights[0], (1 - pred_value) * weights[1], weights[2]]

        max_index = np.argmax(new_weights)
        if max_index == 0:
            interpretations.append(f'b{i+1}')
            continue
        elif max_index == 1:
            interpretations.append(f'not(b{i+1})')
            continue
        else:
            interpretations.append('1')

    return interpretations

def interpret_neuron_weights(Weights):
    """
    Interpret the weights of neurons and return a list of 'b_{i}', 'not(b_{i})', or '1'.

    Args:
        weights (numpy.ndarray): A 2D numpy array where each row corresponds to the weights of a neuron.

    Returns:
        list: A list of strings ('b_{i}', 'not(b_{i})', or '1') for each neuron.
    """
    

    interpretations = []
    for i, weights in enumerate(Weights):

        # compute {weights[0], weights[1], weights[2]}
        new_weights = [weights[0], weights[1], weights[2]]

        max_index = np.argmax(new_weights)
        if max_index == 0:
            interpretations.append(f'b{i+1}')
            continue
        elif max_index == 1:
            interpretations.append(f'not(b{i+1})')
            continue
        else:
            interpretations.append('1')
    return interpretations

## ANDRE Network

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy
import matplotlib.pyplot as plt
BETA = 10.0  # Example value for beta
# Define CustomNeuron and ANDRENet
class CustomNeuron(nn.Module):
    def __init__(self):
        super(CustomNeuron, self).__init__()
        self.raw_weights = nn.Parameter(torch.randn(3) * 0.1)  # Small init to prevent saturation

    def forward(self, x):
        self.weights = torch.softmax(self.raw_weights, dim=0)
        x1 = x
        x2 = 1 - x
        x3 = torch.ones_like(x)
        X1 = self.weights[0] * x1
        X2 = self.weights[1] * x2
        X3 = self.weights[2] * x3
        activated_input = X1 + X2 + X3 - (X1 * X2 + X1 * X3 + X2 * X3) + (X1 * X2 * X3)
        return activated_input

class ANDRENet(nn.Module):
    def __init__(self, num_rules, num_neurons_per_rule, input_dim):
        super(ANDRENet, self).__init__()
        self.num_rules = num_rules
        self.num_neurons = num_neurons_per_rule
        self.input_dim = input_dim
        self.rules = nn.ModuleList([
            nn.ModuleList([CustomNeuron() for _ in range(num_neurons_per_rule)])
            for _ in range(num_rules)
        ])

    def forward(self, x):
        self.saved_weights = []
        self.entropies = []
        rule_outputs = []

        for rule in self.rules:
            conjunctions = []
            rule_weights = []
            entropy_sum = 0.0
            for i, neuron in enumerate(rule):
                xi = x[:, [i % self.input_dim]]
                conjunctions.append(neuron(xi))
                weights = neuron.weights
                rule_weights.append(weights)
                entropy_sum += -(weights * torch.log(weights + 1e-6)).sum()
            rule_vector = torch.cat(rule_weights)
            self.saved_weights.append(rule_vector)
            self.entropies.append(entropy_sum)
            # Soft min of neuron outputs using attention mechanism
            # find -beta * outputs for all neurons, where beta is a hyperparameter
            attention_weights = torch.softmax(-BETA * torch.stack(conjunctions), dim=0)
            outputs = [attention_weights[i] * conjunctions[i] for i in range(self.num_neurons)]
            activated_outputs = torch.sigmoid(5*(sum(outputs) - 0.5))  # Apply sigmoid to the sum
            rule_outputs.append(activated_outputs)

        # Fuzzy disjunction: Soft max of rule outputs using attention mechanism
        # find BETA * rule_outputs for all rules, where beta is a hyperparameter
        attention_weights = torch.softmax(BETA * torch.stack(rule_outputs), dim=0)
        rule_outputs = [attention_weights[i] * rule_outputs[i] for i in range(self.num_rules)]
        activated_output = torch.sigmoid(5*(sum(rule_outputs) - 0.5))  # Apply sigmoid to the sum

        return activated_output

## DFORL Network

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Step 2: DFORLNet Model
class DFORLNet(nn.Module):
    def __init__(self, num_rules=1, input_dim=4):
        super(DFORLNet, self).__init__()
        self.num_rules = num_rules
        self.input_dim = input_dim
        # Initialize raw weights
        self.raw_weights = nn.Parameter(torch.randn(num_rules, input_dim) * 0.1)

    def forward(self, x):
        # Apply softmax to the weights to ensure they are non-negative
        self.shv = torch.softmax(self.raw_weights, dim=1)
        # Rule-wise activation and disjunction (product t-norm)
        act = torch.sigmoid(torch.matmul(self.shv, x.T) - 1)  # shape: (num_rules, batch)
        prod = torch.prod(1 - act, dim=0)
        out = 1 - prod
        return out.unsqueeze(1)

### Automatically Create Dataset 

### Train the models under for different predefined specifications

### Save the extracted rules and the accuracy of each model

In [11]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import random
import re

def parse_target_rule(rule, input_size):
    """
    Parse the target rule and return a function to compute the binary target output.

    Supports logical AND with ',' and logical OR with ';' using bitwise logic.

    Args:
        rule (str): Rule in format like "h :- b1, not(b2); b3, b4."
        input_size (int): Number of input features.

    Returns:
        function: A function that takes input tensor X and returns binary target tensor y.
    """
    # Extract the rule body (everything after 'h :-' and before '.')
    match = re.match(r"h\s*:-\s*(.*)\.", rule.strip())
    if not match:
        raise ValueError(f"Invalid rule format: {rule}")

    or_clauses = match.group(1).split(";")  # OR-separated groups

    def compute_target_output(X):
        X_bin = (X > 0.5)  # Convert to binary (boolean) tensor
        y_total = torch.zeros(X.shape[0], dtype=torch.bool)

        for clause in or_clauses:
            and_clauses = clause.split(",")
            y_clause = torch.ones(X.shape[0], dtype=torch.bool)

            for predicate in and_clauses:
                predicate = predicate.strip()
                if predicate.startswith("not(b"):
                    index = int(predicate[5:-1]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= ~X_bin[:, index]
                elif predicate.startswith("b"):
                    index = int(predicate[1:]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= X_bin[:, index]
                else:
                    raise ValueError(f"Unknown predicate format: {predicate}")

            y_total |= y_clause  # OR across clauses

        return y_total.float()  # Return as float for compatibility with BCE loss etc.

    return compute_target_output

def run_experiment(spec):
    # Extract experiment parameters
    num_samples = spec["Sample Size"]
    input_size = 10
    seed = 42
    data_type = "Fuzzy"
    # data_type = "Boolean"
    noise_percentage = 0 # spec["Noise Percentage"]
    target_rule = spec["Desired Rule"]
    target_rule_size = len(target_rule.split(","))  # Number of predicates in the target rule

    # print(f"Running experiment with parameters: {spec}")

    # Set random seed
    torch.manual_seed(seed)
    # np.random.seed(seed)
    # random.seed(seed)

    # Generate data
    if data_type == "Boolean" or data_type == "boolean":
        X = (torch.rand(num_samples, input_size) > 0.5).float()
    else:  # Fuzzy
        X = torch.rand(num_samples, input_size)

    if target_rule == "Rule1":
        target_rule = "h :- b1, not(b9)."
    elif target_rule == "Rule2":
        target_rule = "h :- b1, not(b9); not(b2), b8."
    elif target_rule == "Rule3":
        target_rule = "h :- b1, not(b9); not(b2), b8;b3, b7."

    Num_rules = target_rule.count(";") + 1

    # Parse the target rule and compute the outputs
    compute_target_output = parse_target_rule(target_rule, input_size)
    y = compute_target_output(X)
    y = (y > 0.5).float()  # Convert to binary output

    # Add noise
    if noise_percentage > 0:
        num_noisy_samples = int(noise_percentage * num_samples / 100)
        noisy_indices = torch.randperm(num_samples)[:num_noisy_samples]
        y[noisy_indices] = 1 - y[noisy_indices]

    # Train ANDRE model
    for i in range(1):
        andre_model = ANDRENet(num_rules=Num_rules, num_neurons_per_rule=input_size, input_dim=input_size)
        andre_optimizer = optim.Adam(andre_model.parameters(), lr=0.01)
        andre_criterion = nn.BCELoss()
        Epochs = 1000

        # Create a data loader
        dataset = torch.utils.data.TensorDataset(X, y)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=len(X), shuffle=True)
        # Train the model
        for epoch in range(Epochs):
            epoch_loss = 0
            for batch_X, batch_y in dataloader:
                andre_optimizer.zero_grad()
                outputs = andre_model(batch_X).squeeze()
                loss = andre_criterion(outputs, batch_y.view(-1))
                loss.backward()
                andre_optimizer.step()
                epoch_loss += loss.item()

        # Evaluate the model (optional)
        with torch.no_grad():
            predictions = (andre_model(X).squeeze() > 0.5).float()
            andre_train_accuracy = (predictions == y).float().mean().item()
        # Extract rule from the trained model
        # Save final weights for inspection
        Weights_nsre = []
        for rule in andre_model.rules:
            rule_weights = []
            for neuron in rule:
                weights = torch.softmax(neuron.raw_weights, dim=0).detach().numpy().tolist()
                rule_weights.append(weights)
            Weights_nsre.append(rule_weights)

        # print("\nFinal Weights of the Network:")
        # for rule_num, rule_weights in enumerate(Weights_nsre):
        #     print(f"Rule {rule_num+1}:")
        #     for i, weights in enumerate(rule_weights):
        #         print(f"  Neuron {i+1}: {weights}") 

        # Rule interpretation
        Predicate_values = X

        Final_rules = []
        for rule_num, weights in enumerate(Weights_nsre):
            Extracted_rules = []
            for i in range(len(Predicate_values)):
                pred_values = Predicate_values[i].numpy()
                preds = interpret_neuron_weights_(pred_values, weights)
                refined_preds = [pred for pred in preds if pred != '1']
                rule = f"h{rule_num+1} :- " + " and ".join(refined_preds) + "."
                Extracted_rules.append(rule)

            rule_counts = Counter(Extracted_rules)
            most_common_rule, count = rule_counts.most_common(1)[0]
            print(f"The most repeated rule is: '{most_common_rule}'")
            print(f"Accuracy over training data: {float(count/len(Extracted_rules))}.")

            Final_rules.append(most_common_rule)

        # remove the "h{i+1} :- " part from each rule
        Final_rules = [rule.replace(f"h{rule_num+1} :- ", "") for rule_num, rule in enumerate(Final_rules)]

        # remove the "."
        Final_rules = [rule.replace(".", "") for rule in Final_rules]

        # convert the rules to a single rule
        final_rule = "h :- " + " OR ".join(Final_rules) + "."
        print("\n======================================")
        print("Final Rule:")
        print(final_rule)
        print("======================================")
        extracted_rule = final_rule
        # Testing the model
        num_test_samples = num_samples
        if data_type == "Boolean" or data_type == "boolean":
            X_test = (torch.rand(num_test_samples, input_size) > 0.5).float()
        else:  # Fuzzy
            X_test = torch.rand(num_test_samples, input_size)
        y_test = compute_target_output(X_test)
        y_test = (y_test > 0.5).float()

        with torch.no_grad():
            andre_predictions = (andre_model(X_test).squeeze() > 0.5).float()
            andre_accuracy_test = (andre_predictions == y_test).float().mean().item()

        # if andre_train_accuracy > 0.8:
        #     print(f"The most repeated rule is: '{most_common_rule}'")
        #     print(f"It appears {count} times out of {len(Extracted_rules)}.")
        #     break


    # Train DFORL model
    dforl_model = DFORLNet(num_rules=Num_rules, input_dim=input_size)
    optimizer = optim.SGD(dforl_model.parameters(), lr=0.5)
    loss_fn = nn.BCELoss()
    lambda_sim = 0.1  # Weight for similarity loss (adjust as needed)

    losses = []
    for epoch in range(Epochs):
        optimizer.zero_grad()
        output = dforl_model(X)
        
        # BCE Loss
        loss_bce = loss_fn(output.squeeze(), y.view(-1))

        # Rule similarity loss (to encourage diversity)
        sim_loss = 0.0
        if Num_rules > 1:
            for i in range(Num_rules):
                for j in range(i + 1, Num_rules):
                    # Cosine similarity between rule i and j
                    sim = torch.cosine_similarity(dforl_model.shv[i], dforl_model.shv[j], dim=0)
                    sim_loss += sim ** 2  # Penalize similarity (target: sim → 0)

        total_loss = loss_bce + lambda_sim * sim_loss
        total_loss.backward()
        optimizer.step()

    # Evaluate DFORL model on training data
    with torch.no_grad():
        dforl_predictions = (dforl_model(X).squeeze() > 0.5).float()
        dforl_accuracy_train = (dforl_predictions == y).float().mean().item()
    
    
    # Evaluate DFORL model
    with torch.no_grad():
        dforl_predictions = (dforl_model(X_test).squeeze() > 0.5).float()
        dforl_accuracy_test = (dforl_predictions == y_test).float().mean().item()

    # Print results
    print(f"NSRE Train Accuracy: {andre_train_accuracy:.4f}")
    print(f"NSRE Test Accuracy: {andre_accuracy_test:.4f}")
    print(f"DFORL Train Accuracy: {dforl_accuracy_train:.4f}")
    print(f"DFORL Test Accuracy: {dforl_accuracy_test:.4f}")
    print(f"Extracted Rule: {extracted_rule}")
    print(f"Target Rule: {target_rule}\n\n")

    return {
        "Sample Size": num_samples,
        "Input Size": input_size,
        "Data Type": data_type,
        "Noise Percentage": noise_percentage,

        "NSRE Accuracy - Train": andre_train_accuracy,
        "NSRE Accuracy - Test": andre_accuracy_test,
        "DFORL Accuracy - Train": dforl_accuracy_train,
        "DFORL Accuracy - Test": dforl_accuracy_test,
        "Extracted Rule": extracted_rule,
        "Target Rule": target_rule,
    }


# Main function to run all experiments
def main():
    # Read experiment specifications from Excel
    specs = pd.read_excel("NSDRE_Comparison_Table2.xlsx")

    # Run experiments
    results = []
    for _, spec in specs.iterrows():
        result = run_experiment(spec)
        results.append(result)

    # Save results to Excel
    results_df = pd.DataFrame(results)
    results_df.to_excel("NSDRE_Comparison_Results.xlsx", index=False)


if __name__ == "__main__":
    main()

The most repeated rule is: 'h1 :- b1 and b7 and not(b8) and not(b9).'
Accuracy over training data: 1.0.

Final Rule:
h :- b1 and b7 and not(b8) and not(b9).
NSRE Train Accuracy: 0.9500
NSRE Test Accuracy: 0.8500
DFORL Train Accuracy: 0.7500
DFORL Test Accuracy: 0.8000
Extracted Rule: h :- b1 and b7 and not(b8) and not(b9).
Target Rule: h :- b1, not(b9).


The most repeated rule is: 'h1 :- b1 and not(b9).'
Accuracy over training data: 1.0.

Final Rule:
h :- b1 and not(b9).
NSRE Train Accuracy: 0.9800
NSRE Test Accuracy: 0.9600
DFORL Train Accuracy: 0.7200
DFORL Test Accuracy: 0.6600
Extracted Rule: h :- b1 and not(b9).
Target Rule: h :- b1, not(b9).


The most repeated rule is: 'h1 :- b1 and not(b9).'
Accuracy over training data: 1.0.

Final Rule:
h :- b1 and not(b9).
NSRE Train Accuracy: 0.9500
NSRE Test Accuracy: 0.9700
DFORL Train Accuracy: 0.7300
DFORL Test Accuracy: 0.8000
Extracted Rule: h :- b1 and not(b9).
Target Rule: h :- b1, not(b9).


The most repeated rule is: 'h1 :- b1 and

## Curriculum Learning

### Synthetic Dataset without Noise

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import random
import re

from alphailp.src import optimizer

def parse_target_rule(rule, input_size):
    """
    Parse the target rule and return a function to compute the binary target output.

    Supports logical AND with ',' and logical OR with ';' using bitwise logic.

    Args:
        rule (str): Rule in format like "h :- b1, not(b2); b3, b4."
        input_size (int): Number of input features.

    Returns:
        function: A function that takes input tensor X and returns binary target tensor y.
    """
    # Extract the rule body (everything after 'h :-' and before '.')
    match = re.match(r"h\s*:-\s*(.*)\.", rule.strip())
    if not match:
        raise ValueError(f"Invalid rule format: {rule}")

    or_clauses = match.group(1).split(";")  # OR-separated groups

    def compute_target_output(X):
        X_bin = (X > 0.5)  # Convert to binary (boolean) tensor
        y_total = torch.zeros(X.shape[0], dtype=torch.bool)

        for clause in or_clauses:
            and_clauses = clause.split(",")
            y_clause = torch.ones(X.shape[0], dtype=torch.bool)

            for predicate in and_clauses:
                predicate = predicate.strip()
                if predicate.startswith("not(b"):
                    index = int(predicate[5:-1]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= ~X_bin[:, index]
                elif predicate.startswith("b"):
                    index = int(predicate[1:]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= X_bin[:, index]
                else:
                    raise ValueError(f"Unknown predicate format: {predicate}")

            y_total |= y_clause  # OR across clauses

        return y_total.float()  # Return as float for compatibility with BCE loss etc.

    return compute_target_output

# Function to freeze a rule
def freeze_rule(model, rule_index):
    for param in model.rules[rule_index].parameters():
        param.requires_grad = False

def run_experiment(spec):
    # Extract experiment parameters
    num_samples = spec["Sample Size"]
    input_size = 10
    seed = 42
    data_type = "Fuzzy"
    noise_percentage = 0 # spec["Noise Percentage"]
    target_rule = spec["Desired Rule"]
    target_rule_size = len(target_rule.split(","))  # Number of predicates in the target rule

    # print(f"Running experiment with parameters: {spec}")

    # Set random seed
    torch.manual_seed(seed)
    # np.random.seed(seed)
    # random.seed(seed)

    # Generate data
    if data_type == "Boolean" or data_type == "boolean":
        X_train = (torch.rand(num_samples, input_size) > 0.5).float()
        X_train[X_train == 0] = 0.1  # Ensure no zero values for boolean data
    else:  # Fuzzy
        X_train = torch.rand(num_samples, input_size)

    if target_rule == "Rule1":
        target_rule = "h :- b1, not(b9)."
    elif target_rule == "Rule2":
        target_rule = "h :- b1, not(b9); not(b2), b8."
    elif target_rule == "Rule3":
        target_rule = "h :- b1, not(b9); not(b2), b8; b3, not(b5), b7."

    Num_rules = target_rule.count(";") + 1

    # Parse the target rule and compute the outputs
    compute_target_output = parse_target_rule(target_rule, input_size)
    y_train = compute_target_output(X_train)
    y_train = (y_train > 0.5).float()  # Convert to binary output

    # Add noise
    if noise_percentage > 0:
        num_noisy_samples = int(noise_percentage * num_samples / 100)
        noisy_indices = torch.randperm(num_samples)[:num_noisy_samples]
        y_train[noisy_indices] = 1 - y_train[noisy_indices]

    for i in range(1):
        # Train NSRE model
        # torch.manual_seed(seed)
        # nsre_model = CustomNeuralNet(num_neurons=input_size, input_dim=input_size)
        # andre_model = ANDRENet(num_rules=Num_rules, num_neurons_per_rule=input_size, input_dim=input_size)
        # andre_optimizer = optim.Adam(andre_model.parameters(), lr=0.01)
        # andre_criterion = nn.BCELoss()

        # Create a data loader
        # dataset = torch.utils.data.TensorDataset(X, y)
        # dataloader = torch.utils.data.DataLoader(dataset, batch_size=len(X), shuffle=True)
        # # Train the model
        # for epoch in range(Epochs):
        #     epoch_loss = 0
        #     for batch_X, batch_y in dataloader:
        #         andre_optimizer.zero_grad()
        #         outputs = andre_model(batch_X).squeeze()
        #         loss = andre_criterion(outputs, batch_y)
        #         loss.backward()
        #         andre_optimizer.step()
        #         epoch_loss += loss.item()


        frozen_rule_states = []
        num_restarts = 3
        Epochs = 1000

        # Curriculum Learning Loop
        for rule_index in range(Num_rules):  # Learn 3 rules step-by-step
            print(f"\n=== Training Rule {rule_index + 1} ===")

            # Cosine similarity function
            cosine = nn.CosineSimilarity(dim=1)

            num_rules = rule_index + 1
            best_state_dict = None
            best_loss = float('inf')
            best_losses = []

            for restart in range(num_restarts):
                andre_model = ANDRENet(num_rules=num_rules, num_neurons_per_rule=input_size, input_dim=input_size)

                # Load and freeze previous rules
                if frozen_rule_states:
                    andre_model.load_state_dict(frozen_rule_states[-1], strict=False)
                    for i in range(rule_index):
                        freeze_rule(andre_model, i)

                andre_criterion = nn.BCELoss()
                andre_optimizer = optim.AdamW(filter(lambda p: p.requires_grad, andre_model.parameters()), lr=0.01, weight_decay=1e-4)
                # scheduler = optim.lr_scheduler.StepLR(andre_optimizer, step_size=250, gamma=0.5)

                losses_nsre = []

                for epoch in range(Epochs):
                    andre_optimizer.zero_grad()
                    outputs = andre_model(X_train).squeeze()
                    bce_loss = andre_criterion(outputs, y_train)

                    penalty = 0.0
                    for i in range(num_rules):
                        for j in range(i + 1, num_rules):
                            wi = andre_model.saved_weights[i].view(input_size, -1)
                            wj = andre_model.saved_weights[j].view(input_size, -1)
                            pairwise_sim = cosine(wi, wj).mean()
                            penalty += pairwise_sim

                    entropy_term = sum(andre_model.entropies)
                    total_loss = bce_loss #+ similarity_weight * penalty - entropy_weight * entropy_term
                    total_loss.backward()
                    nn.utils.clip_grad_norm_(andre_model.parameters(), max_norm=1.0)
                    andre_optimizer.step()
                    # scheduler.step()
                    losses_nsre.append(total_loss.item())


                final_loss = losses_nsre[-1]
                print(f"Restart {restart + 1}/{num_restarts}, Final Loss: {final_loss:.4f}")
                if final_loss < best_loss:
                    best_loss = final_loss
                    best_losses = losses_nsre
                    best_state_dict = deepcopy(andre_model.state_dict())

            frozen_rule_states.append(best_state_dict)
        # Evaluate the model (optional)
        with torch.no_grad():
            predictions = (andre_model(X_train).squeeze() > 0.5).float()
            andre_train_accuracy = (predictions == y_train).float().mean().item()
        # Extract rule from the trained model
        # Corrected code

        # Save final weights for inspection
        Weights_nsre = []
        for rule in andre_model.rules:
            rule_weights = []
            for neuron in rule:
                weights = torch.softmax(neuron.raw_weights, dim=0).detach().numpy().tolist()
                rule_weights.append(weights)
            Weights_nsre.append(rule_weights)

        # print("\nFinal Weights of the Network:")
        # for rule_num, rule_weights in enumerate(Weights_nsre):
        #     print(f"Rule {rule_num+1}:")
        #     for i, weights in enumerate(rule_weights):
        #         print(f"  Neuron {i+1}: {weights}") 

        # Rule interpretation
        Predicate_values = X_train

        Final_rules = []
        for rule_num, weights in enumerate(Weights_nsre):
            Extracted_rules = []
            for i in range(len(Predicate_values)):
                pred_values = Predicate_values[i].numpy()
                preds = interpret_neuron_weights_(pred_values, weights)
                refined_preds = [pred for pred in preds if pred != '1']
                rule = f"h{rule_num+1} :- " + " and ".join(refined_preds) + "."
                Extracted_rules.append(rule)

            rule_counts = Counter(Extracted_rules)
            most_common_rule, count = rule_counts.most_common(1)[0]
            print(f"The most repeated rule is: '{most_common_rule}'")
            print(f"Accuracy over training data: {float(count/len(Extracted_rules))}.")

            Final_rules.append(most_common_rule)

        # remove the "h{i+1} :- " part from each rule
        Final_rules = [rule.replace(f"h{rule_num+1} :- ", "") for rule_num, rule in enumerate(Final_rules)]

        # remove the "."
        Final_rules = [rule.replace(".", "") for rule in Final_rules]

        # convert the rules to a single rule
        final_rule = "h :- " + " OR ".join(Final_rules) + "."
        print("\n======================================")
        print("Final Rule:")
        print(final_rule)
        print("======================================")
        extracted_rule = final_rule
        # Testing the model
        num_test_samples = num_samples
        if data_type == "Boolean" or data_type == "boolean":
            X_test = (torch.rand(num_test_samples, input_size) > 0.5).float()
        else:  # Fuzzy
            X_test = torch.rand(num_test_samples, input_size)
        y_test = compute_target_output(X_test)
        y_test = (y_test > 0.5).float()

        with torch.no_grad():
            andre_predictions = (andre_model(X_test).squeeze() > 0.5).float()
            andre_accuracy_test = (andre_predictions == y_test).float().mean().item()

        # if andre_train_accuracy > 0.8:
        #     print(f"The most repeated rule is: '{most_common_rule}'")
        #     print(f"It appears {count} times out of {len(Extracted_rules)}.")
        #     break


    # Train DFORL model
    dforl_model = DFORLNet(num_rules=Num_rules, input_dim=input_size)
    optimizer = optim.SGD(dforl_model.parameters(), lr=0.5)
    loss_fn = nn.BCELoss()
    lambda_sim = 0.1  # Weight for similarity loss (adjust as needed)

    losses = []
    for epoch in range(Epochs):
        optimizer.zero_grad()
        output = dforl_model(X_train)
        
        # BCE Loss
        loss_bce = loss_fn(output.squeeze(), y_train.view(-1))

        # Rule similarity loss (to encourage diversity)
        sim_loss = 0.0
        if Num_rules > 1:
            for i in range(Num_rules):
                for j in range(i + 1, Num_rules):
                    # Cosine similarity between rule i and j
                    sim = torch.cosine_similarity(dforl_model.shv[i], dforl_model.shv[j], dim=0)
                    sim_loss += sim ** 2  # Penalize similarity (target: sim → 0)

        total_loss = loss_bce + lambda_sim * sim_loss
        total_loss.backward()
        optimizer.step()

    # Evaluate DFORL model on training data
    with torch.no_grad():
        dforl_predictions = (dforl_model(X_train).squeeze() > 0.5).float()
        dforl_accuracy_train = (dforl_predictions == y_train).float().mean().item()
    
    
    # Evaluate DFORL model
    with torch.no_grad():
        dforl_predictions = (dforl_model(X_test).squeeze() > 0.5).float()
        dforl_accuracy_test = (dforl_predictions == y_test).float().mean().item()

    # Print results
    print(f"NSRE Train Accuracy: {andre_train_accuracy:.4f}")
    print(f"NSRE Test Accuracy: {andre_accuracy_test:.4f}")
    print(f"DFORL Train Accuracy: {dforl_accuracy_train:.4f}")
    print(f"DFORL Test Accuracy: {dforl_accuracy_test:.4f}")
    print(f"Extracted Rule: {extracted_rule}")
    print(f"Target Rule: {target_rule}\n\n")

    return {
        "Sample Size": num_samples,
        "Input Size": input_size,
        "Data Type": data_type,
        "Noise Percentage": noise_percentage,

        "NSRE Accuracy - Train": andre_train_accuracy,
        "NSRE Accuracy - Test": andre_accuracy_test,
        "DFORL Accuracy - Train": dforl_accuracy_train,
        "DFORL Accuracy - Test": dforl_accuracy_test,
        "Extracted Rule": extracted_rule,
        "Target Rule": target_rule,
    }


# Main function to run all experiments
def main():
    # Read experiment specifications from Excel
    specs = pd.read_excel("NSDRE_Comparison_Table2.xlsx")

    # Run experiments
    results = []
    for _, spec in specs.iterrows():
        result = run_experiment(spec)
        results.append(result)

    # Save results to Excel
    results_df = pd.DataFrame(results)
    results_df.to_excel("NSDRE_Comparison_Results.xlsx", index=False)


if __name__ == "__main__":
    main()


=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.2580
Restart 2/3, Final Loss: 0.2581
Restart 3/3, Final Loss: 0.2617
The most repeated rule is: 'h1 :- b1 and b3 and b7 and not(b8) and not(b9).'
Accuracy over training data: 1.0.

Final Rule:
h :- b1 and b3 and b7 and not(b8) and not(b9).
NSRE Train Accuracy: 0.9500
NSRE Test Accuracy: 0.7500
DFORL Train Accuracy: 0.7500
DFORL Test Accuracy: 0.7500
Extracted Rule: h :- b1 and b3 and b7 and not(b8) and not(b9).
Target Rule: h :- b1, not(b9).



=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.2900
Restart 2/3, Final Loss: 0.2899
Restart 3/3, Final Loss: 0.2901
The most repeated rule is: 'h1 :- b1 and not(b9).'
Accuracy over training data: 1.0.

Final Rule:
h :- b1 and not(b9).
NSRE Train Accuracy: 0.9800
NSRE Test Accuracy: 0.9000
DFORL Train Accuracy: 0.7200
DFORL Test Accuracy: 0.7600
Extracted Rule: h :- b1 and not(b9).
Target Rule: h :- b1, not(b9).



=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.2990
Restart 2/3, Final Lo

### Synthetic Dataset with Noise

In [6]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import random
import re

from alphailp.src import optimizer

def parse_target_rule(rule, input_size):
    """
    Parse the target rule and return a function to compute the binary target output.

    Supports logical AND with ',' and logical OR with ';' using bitwise logic.

    Args:
        rule (str): Rule in format like "h :- b1, not(b2); b3, b4."
        input_size (int): Number of input features.

    Returns:
        function: A function that takes input tensor X and returns binary target tensor y.
    """
    # Extract the rule body (everything after 'h :-' and before '.')
    match = re.match(r"h\s*:-\s*(.*)\.", rule.strip())
    if not match:
        raise ValueError(f"Invalid rule format: {rule}")

    or_clauses = match.group(1).split(";")  # OR-separated groups

    def compute_target_output(X):
        X_bin = (X > 0.5)  # Convert to binary (boolean) tensor
        y_total = torch.zeros(X.shape[0], dtype=torch.bool)

        for clause in or_clauses:
            and_clauses = clause.split(",")
            y_clause = torch.ones(X.shape[0], dtype=torch.bool)

            for predicate in and_clauses:
                predicate = predicate.strip()
                if predicate.startswith("not(b"):
                    index = int(predicate[5:-1]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= ~X_bin[:, index]
                elif predicate.startswith("b"):
                    index = int(predicate[1:]) - 1
                    if index >= input_size:
                        raise ValueError(f"Predicate index {index + 1} exceeds input size {input_size}")
                    y_clause &= X_bin[:, index]
                else:
                    raise ValueError(f"Unknown predicate format: {predicate}")

            y_total |= y_clause  # OR across clauses

        return y_total.float()  # Return as float for compatibility with BCE loss etc.

    return compute_target_output

# Function to freeze a rule
def freeze_rule(model, rule_index):
    for param in model.rules[rule_index].parameters():
        param.requires_grad = False

def run_experiment(spec):
    # Extract experiment parameters
    num_samples = spec["Sample Size"]
    input_size = 10
    seed = 42
    data_type = "Fuzzy"
    noise_percentage = spec["Noise Percentage"]
    target_rule = spec["Desired Rule"]
    target_rule_size = len(target_rule.split(","))  # Number of predicates in the target rule

    # print(f"Running experiment with parameters: {spec}")

    # Set random seed
    torch.manual_seed(seed)
    # np.random.seed(seed)
    # random.seed(seed)

    # Generate data
    if data_type == "Boolean" or data_type == "boolean":
        X_train = (torch.rand(num_samples, input_size) > 0.5).float()
        X_train[X_train == 0] = 0.1  # Ensure no zero values for boolean data
    else:  # Fuzzy
        X_train = torch.rand(num_samples, input_size)

    if target_rule == "Rule4":
        target_rule = "h :- not(b1), b9."
    elif target_rule == "Rule5":
        target_rule = "h :- not(b1), b9; b2, not(b8)."
    elif target_rule == "Rule6":
        target_rule = "h :- not(b1), b9; b2, not(b8); not(b3), b5, not(b7)."

    Num_rules = target_rule.count(";") + 1

    # Parse the target rule and compute the outputs
    compute_target_output = parse_target_rule(target_rule, input_size)
    y_train = compute_target_output(X_train)
    y_train = (y_train > 0.5).float()  # Convert to binary output

    # Add noise
    if noise_percentage > 0:
        num_noisy_samples = int(noise_percentage * num_samples / 100)
        noisy_indices = torch.randperm(num_samples)[:num_noisy_samples]
        y_train[noisy_indices] = 1 - y_train[noisy_indices]

    for i in range(1):
        # Train NSRE model
        # torch.manual_seed(seed)
        # nsre_model = CustomNeuralNet(num_neurons=input_size, input_dim=input_size)
        # andre_model = ANDRENet(num_rules=Num_rules, num_neurons_per_rule=input_size, input_dim=input_size)
        # andre_optimizer = optim.Adam(andre_model.parameters(), lr=0.01)
        # andre_criterion = nn.BCELoss()

        # Create a data loader
        # dataset = torch.utils.data.TensorDataset(X, y)
        # dataloader = torch.utils.data.DataLoader(dataset, batch_size=len(X), shuffle=True)
        # # Train the model
        # for epoch in range(Epochs):
        #     epoch_loss = 0
        #     for batch_X, batch_y in dataloader:
        #         andre_optimizer.zero_grad()
        #         outputs = andre_model(batch_X).squeeze()
        #         loss = andre_criterion(outputs, batch_y)
        #         loss.backward()
        #         andre_optimizer.step()
        #         epoch_loss += loss.item()


        frozen_rule_states = []
        num_restarts = 3
        Epochs = 1000

        # Curriculum Learning Loop
        for rule_index in range(Num_rules):  # Learn 3 rules step-by-step
            print(f"\n=== Training Rule {rule_index + 1} ===")

            # Cosine similarity function
            cosine = nn.CosineSimilarity(dim=1)

            num_rules = rule_index + 1
            best_state_dict = None
            best_loss = float('inf')
            best_losses = []

            for restart in range(num_restarts):
                andre_model = ANDRENet(num_rules=num_rules, num_neurons_per_rule=input_size, input_dim=input_size)

                # Load and freeze previous rules
                if frozen_rule_states:
                    andre_model.load_state_dict(frozen_rule_states[-1], strict=False)
                    for i in range(rule_index):
                        freeze_rule(andre_model, i)

                andre_criterion = nn.BCELoss()
                andre_optimizer = optim.AdamW(filter(lambda p: p.requires_grad, andre_model.parameters()), lr=0.01, weight_decay=1e-4)
                # scheduler = optim.lr_scheduler.StepLR(andre_optimizer, step_size=250, gamma=0.5)

                losses_nsre = []

                for epoch in range(Epochs):
                    andre_optimizer.zero_grad()
                    outputs = andre_model(X_train).squeeze()
                    bce_loss = andre_criterion(outputs, y_train)

                    penalty = 0.0
                    for i in range(num_rules):
                        for j in range(i + 1, num_rules):
                            wi = andre_model.saved_weights[i].view(input_size, -1)
                            wj = andre_model.saved_weights[j].view(input_size, -1)
                            pairwise_sim = cosine(wi, wj).mean()
                            penalty += pairwise_sim

                    entropy_term = sum(andre_model.entropies)
                    total_loss = bce_loss #+ similarity_weight * penalty - entropy_weight * entropy_term
                    total_loss.backward()
                    nn.utils.clip_grad_norm_(andre_model.parameters(), max_norm=1.0)
                    andre_optimizer.step()
                    # scheduler.step()
                    losses_nsre.append(total_loss.item())


                final_loss = losses_nsre[-1]
                print(f"Restart {restart + 1}/{num_restarts}, Final Loss: {final_loss:.4f}")
                if final_loss < best_loss:
                    best_loss = final_loss
                    best_losses = losses_nsre
                    best_state_dict = deepcopy(andre_model.state_dict())

            frozen_rule_states.append(best_state_dict)
        # Evaluate the model (optional)
        with torch.no_grad():
            predictions = (andre_model(X_train).squeeze() > 0.5).float()
            andre_train_accuracy = (predictions == y_train).float().mean().item()
        # Extract rule from the trained model
        # Corrected code

        # Save final weights for inspection
        Weights_nsre = []
        for rule in andre_model.rules:
            rule_weights = []
            for neuron in rule:
                weights = torch.softmax(neuron.raw_weights, dim=0).detach().numpy().tolist()
                rule_weights.append(weights)
            Weights_nsre.append(rule_weights)

        # print("\nFinal Weights of the Network:")
        # for rule_num, rule_weights in enumerate(Weights_nsre):
        #     print(f"Rule {rule_num+1}:")
        #     for i, weights in enumerate(rule_weights):
        #         print(f"  Neuron {i+1}: {weights}") 

        # Rule interpretation
        Predicate_values = X_train

        Final_rules = []
        for rule_num, weights in enumerate(Weights_nsre):
            Extracted_rules = []
            for i in range(len(Predicate_values)):
                pred_values = Predicate_values[i].numpy()
                preds = interpret_neuron_weights_(pred_values, weights)
                refined_preds = [pred for pred in preds if pred != '1']
                rule = f"h{rule_num+1} :- " + " and ".join(refined_preds) + "."
                Extracted_rules.append(rule)

            rule_counts = Counter(Extracted_rules)
            most_common_rule, count = rule_counts.most_common(1)[0]
            print(f"The most repeated rule is: '{most_common_rule}'")
            print(f"Accuracy over training data: {float(count/len(Extracted_rules))}.")

            Final_rules.append(most_common_rule)

        # remove the "h{i+1} :- " part from each rule
        Final_rules = [rule.replace(f"h{rule_num+1} :- ", "") for rule_num, rule in enumerate(Final_rules)]

        # remove the "."
        Final_rules = [rule.replace(".", "") for rule in Final_rules]

        # convert the rules to a single rule
        final_rule = "h :- " + " OR ".join(Final_rules) + "."
        print("\n======================================")
        print("Final Rule:")
        print(final_rule)
        print("======================================")
        extracted_rule = final_rule
        # Testing the model
        num_test_samples = num_samples
        if data_type == "Boolean" or data_type == "boolean":
            X_test = (torch.rand(num_test_samples, input_size) > 0.5).float()
        else:  # Fuzzy
            X_test = torch.rand(num_test_samples, input_size)
        y_test = compute_target_output(X_test)
        # Add noise
        if noise_percentage > 0:
            num_noisy_samples = int(noise_percentage * num_test_samples / 100)
            noisy_indices = torch.randperm(num_test_samples)[:num_noisy_samples]
            y_test[noisy_indices] = 1 - y_test[noisy_indices]

        with torch.no_grad():
            andre_predictions = (andre_model(X_test).squeeze() > 0.5).float()
            andre_accuracy_test = (andre_predictions == y_test).float().mean().item()

        # if andre_train_accuracy > 0.8:
        #     print(f"The most repeated rule is: '{most_common_rule}'")
        #     print(f"It appears {count} times out of {len(Extracted_rules)}.")
        #     break


    # Train DFORL model
    dforl_model = DFORLNet(num_rules=Num_rules, input_dim=input_size)
    optimizer = optim.SGD(dforl_model.parameters(), lr=0.5)
    loss_fn = nn.BCELoss()
    lambda_sim = 0.1  # Weight for similarity loss (adjust as needed)

    losses = []
    for epoch in range(Epochs):
        optimizer.zero_grad()
        output = dforl_model(X_train)
        
        # BCE Loss
        loss_bce = loss_fn(output.squeeze(), y_train.view(-1))

        # Rule similarity loss (to encourage diversity)
        sim_loss = 0.0
        if Num_rules > 1:
            for i in range(Num_rules):
                for j in range(i + 1, Num_rules):
                    # Cosine similarity between rule i and j
                    sim = torch.cosine_similarity(dforl_model.shv[i], dforl_model.shv[j], dim=0)
                    sim_loss += sim ** 2  # Penalize similarity (target: sim → 0)

        total_loss = loss_bce + lambda_sim * sim_loss
        total_loss.backward()
        optimizer.step()

    # Evaluate DFORL model on training data
    with torch.no_grad():
        dforl_predictions = (dforl_model(X_train).squeeze() > 0.5).float()
        dforl_accuracy_train = (dforl_predictions == y_train).float().mean().item()
    
    
    # Evaluate DFORL model
    with torch.no_grad():
        dforl_predictions = (dforl_model(X_test).squeeze() > 0.5).float()
        dforl_accuracy_test = (dforl_predictions == y_test).float().mean().item()

    # Print results
    print(f"NSRE Train Accuracy: {andre_train_accuracy:.4f}")
    print(f"NSRE Test Accuracy: {andre_accuracy_test:.4f}")
    print(f"DFORL Train Accuracy: {dforl_accuracy_train:.4f}")
    print(f"DFORL Test Accuracy: {dforl_accuracy_test:.4f}")
    print(f"Extracted Rule: {extracted_rule}")
    print(f"Target Rule: {target_rule}\n\n")

    return {
        "Sample Size": num_samples,
        "Input Size": input_size,
        "Data Type": data_type,
        "Noise Percentage": noise_percentage,

        "NSRE Accuracy - Train": andre_train_accuracy,
        "NSRE Accuracy - Test": andre_accuracy_test,
        "DFORL Accuracy - Train": dforl_accuracy_train,
        "DFORL Accuracy - Test": dforl_accuracy_test,
        "Extracted Rule": extracted_rule,
        "Target Rule": target_rule,
    }


# Main function to run all experiments
def main():
    # Read experiment specifications from Excel
    specs = pd.read_excel("NSDRE_Comparison_Table3.xlsx")

    # Run experiments
    results = []
    for _, spec in specs.iterrows():
        result = run_experiment(spec)
        results.append(result)

    # Save results to Excel
    results_df = pd.DataFrame(results)
    results_df.to_excel("NSDRE_Comparison_Results_noisy.xlsx", index=False)


if __name__ == "__main__":
    main()


=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.4112
Restart 2/3, Final Loss: 0.4111
Restart 3/3, Final Loss: 0.4112
The most repeated rule is: 'h1 :- not(b1) and b9.'
Accuracy over training data: 0.995.

Final Rule:
h :- not(b1) and b9.
NSRE Train Accuracy: 0.8650
NSRE Test Accuracy: 0.8650
DFORL Train Accuracy: 0.7100
DFORL Test Accuracy: 0.6650
Extracted Rule: h :- not(b1) and b9.
Target Rule: h :- not(b1), b9.



=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.5184
Restart 2/3, Final Loss: 0.5187
Restart 3/3, Final Loss: 0.5184
The most repeated rule is: 'h1 :- not(b1) and b9.'
Accuracy over training data: 0.64.

Final Rule:
h :- not(b1) and b9.
NSRE Train Accuracy: 0.7950
NSRE Test Accuracy: 0.7400
DFORL Train Accuracy: 0.6800
DFORL Test Accuracy: 0.6150
Extracted Rule: h :- not(b1) and b9.
Target Rule: h :- not(b1), b9.



=== Training Rule 1 ===
Restart 1/3, Final Loss: 0.6155
Restart 2/3, Final Loss: 0.6217
Restart 3/3, Final Loss: 0.6142
The most repeated rule is: 'h1 :